# 05 – NLP on Hydrological Reports
**AquaML Portfolio Project**

Text analysis of short hydrological situation reports:
1. Supervised severity classification (severe / moderate / normal / drought)
2. Simple topic exploration with TF-IDF + clustering / LDA-style insights

Demonstrates classic + modern NLP pipelines that work well on domain reports.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.cluster import KMeans

from src.evaluation import classification_metrics
from src.visualization import plot_confusion_matrix


In [ ]:
reports = pd.read_csv(ROOT / "data/sample/hydrology_reports.csv")
print(reports["severity"].value_counts())
reports.sample(3, random_state=1)


## Severity Classification

In [ ]:
le = LabelEncoder()
y = le.fit_transform(reports["severity"])
class_names = list(le.classes_)

X_train, X_test, y_train, y_test = train_test_split(
    reports["text"], y, test_size=0.25, stratify=y, random_state=42
)

tfidf = TfidfVectorizer(max_features=2000, ngram_range=(1, 2), stop_words="english", min_df=2)
X_tr = tfidf.fit_transform(X_train)
X_te = tfidf.transform(X_test)

clf = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
clf.fit(X_tr, y_train)
pred = clf.predict(X_te)

print(classification_metrics(y_test, pred))
print(classification_report(y_test, pred, target_names=class_names, digits=3))


In [ ]:
cm = confusion_matrix(y_test, pred)
plot_confusion_matrix(cm, class_names, title="Severity Classification")
plt.show()


## Most indicative words per class

In [ ]:
feature_names = np.array(tfidf.get_feature_names_out())
for i, cls in enumerate(class_names):
    coefs = clf.coef_[i]
    top = feature_names[np.argsort(coefs)[-8:][::-1]]
    print(f"{cls:10s}: {', '.join(top)}")


## Topic Modeling (LDA)

In [ ]:
# Fit LDA on the full corpus
X_all = tfidf.fit_transform(reports["text"])
lda = LatentDirichletAllocation(n_components=4, random_state=42, max_iter=20)
lda.fit(X_all)

print("Top words per topic:")
for topic_idx, topic in enumerate(lda.components_):
    top_words = feature_names[np.argsort(topic)[-10:][::-1]]
    print(f"Topic {topic_idx}: {', '.join(top_words)}")


## Takeaways
- Even short domain reports contain strong lexical signals for severity.
- TF-IDF + linear models remain competitive and highly interpretable.
- Topic models help surface recurring themes (flood response, drought, monitoring, etc.).
- For production: add spaCy NER for locations / magnitudes, or fine-tune a small transformer.
